# Basal Sliding: Lecture Figures

**Ge 193 — Glacier Dynamics, Spring 2026**

This notebook generates figures for the *Basal Sliding* lecture, covering Weertman sliding theory, cavity formation, effective pressure effects, Mohr–Coulomb till behavior, and phenomenological sliding laws.

In [ ]:
# ---- Setup: imports and physical constants ----
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# Physical constants
rho_i = 917.0       # kg m^{-3}, ice density
g = 9.8             # m s^{-2}
k_r = 2.5           # W m^{-1} K^{-1}, thermal conductivity of bedrock
B_cc = 7.4e-8       # K Pa^{-1}, Clausius-Clapeyron constant
L = 3.34e5          # J kg^{-1}, specific latent heat of fusion
rho_w = 1000.0      # kg m^{-3}, water density
sec_per_yr = 3.154e7  # s yr^{-1}

# Glen's flow law parameters
n_glen = 3
A_glen = 2.4e-24     # Pa^{-3} s^{-1} (at T ~ 263 K)

print('Setup complete.')

---
## Figure 1: Regelation and Creep Velocities vs Bump Size

Show how the regelation velocity $u_1 \propto 1/a$ and the enhanced creep velocity $u_2 \propto a$ depend on bump size, and identify the controlling obstacle size $a_c$ where the total velocity is minimized.

In [ ]:
# ---- Fig 1: regelation and creep velocity vs bump size ----

def u_regelation(a, tau_b, R):
    """Regelation velocity (m/s). a in m, tau_b in Pa, R = a/lambda."""
    return k_r * B_cc / (rho_i * L * a) * tau_b / R**2


def u_creep(a, tau_b, R, A=A_glen, n=n_glen):
    """Enhanced creep velocity (m/s). a in m, tau_b in Pa."""
    return 2 * 3**(-(n+1)/2) * a * A * (tau_b / (2 * R**2))**n


tau_b_val = 100e3   # Pa (100 kPa)
R_val = 0.1         # roughness a/lambda

a_arr = np.logspace(-4, 0, 500)  # bump size in m (0.1 mm to 1 m)

u1 = u_regelation(a_arr, tau_b_val, R_val)
u2 = u_creep(a_arr, tau_b_val, R_val)
u_total = u1 + u2

# Controlling obstacle size: where u1 = u2
idx_cross = np.argmin(np.abs(np.log10(u1) - np.log10(u2)))
a_c = a_arr[idx_cross]
u_c = u_total[idx_cross]

fig, ax = plt.subplots(figsize=(7, 5))

ax.loglog(a_arr * 1e3, u1 * sec_per_yr, '--', color='#d62728', lw=2,
          label='Regelation ($u_1 \\propto 1/a$)')
ax.loglog(a_arr * 1e3, u2 * sec_per_yr, '--', color='#1f77b4', lw=2,
          label='Enhanced creep ($u_2 \\propto a^n$)')
ax.loglog(a_arr * 1e3, u_total * sec_per_yr, 'k-', lw=2.5,
          label='Total ($u_1 + u_2$)')

# Mark controlling obstacle size
ax.plot(a_c * 1e3, u_c * sec_per_yr, 'o', color='#ff7f0e', ms=12, zorder=5,
        markeredgecolor='k', markeredgewidth=1)
ax.annotate(f'Controlling obstacle\n$a_c = {a_c*1e3:.0f}$ mm',
            xy=(a_c * 1e3, u_c * sec_per_yr),
            xytext=(a_c * 1e3 * 5, u_c * sec_per_yr * 0.3),
            fontsize=10, ha='left', va='top', color='#ff7f0e',
            arrowprops=dict(arrowstyle='->', color='#ff7f0e', lw=1.5))

ax.set_xlabel('Bump size $a$ (mm)', fontsize=12)
ax.set_ylabel('Sliding velocity (m yr$^{-1}$)', fontsize=12)
ax.set_title(f'$\\tau_b = {tau_b_val/1e3:.0f}$ kPa, $R = {R_val}$', fontsize=12)
ax.set_xlim(0.1, 1000)
ax.legend(loc='upper right', fontsize=10, frameon=False)

fig.tight_layout()
fig.savefig('figures/fig01_regelation_creep_vs_bump_size.pdf', bbox_inches='tight')
plt.show()
print(f'Controlling obstacle size: a_c = {a_c*1e3:.1f} mm')
print('Figure 1 saved.')

---
## Figure 2: Basal Drag vs Slip Velocity (Schoof/Gagliardini Sliding Law)

Show the relationship between basal drag $\tau_b$ and slip velocity $u_b$ for a hard bed with cavities, using the regularized Coulomb sliding law from Schoof (2005) and Gagliardini et al. (2007). This replaces the placeholder figure in the tex file.

The sliding law is:
$$\frac{\tau_b}{N} = C_s \left(\frac{u_b}{u_b + C_s^n N^n A^{-1} \lambda_{\max}}\right)^{1/n}$$

which transitions from Weertman-like ($\tau_b \propto u_b^{1/n}$) at low velocity to Coulomb ($\tau_b \to C_s N$) at high velocity.

In [ ]:
# ---- Fig 2: drag vs slip velocity (regularized Coulomb law) ----

def tau_b_schoof(u_b, N, C_s, n, A, lambda_max):
    """Schoof (2005) / Gagliardini (2007) sliding law.
    u_b in m/yr, N in Pa, returns tau_b in Pa.
    """
    u_b_si = u_b / sec_per_yr  # convert to m/s
    u_0 = C_s**n * N**n / A * lambda_max
    return C_s * N * (u_b_si / (u_b_si + u_0))**(1.0 / n)


def tau_b_weertman(u_b, C_w, m):
    """Weertman sliding law: tau_b = C_w^{-1/m} u_b^{1/m}.
    u_b in m/yr, returns tau_b in Pa.
    """
    u_b_si = u_b / sec_per_yr
    return C_w**(-1.0/m) * u_b_si**(1.0/m)


# Parameters
n_sl = 3
C_s = 0.5       # Iken's bound coefficient (tan beta)
A_sl = 2.4e-24  # Pa^{-3} s^{-1}
lambda_max = 0.5  # dimensionless, max bed slope parameter

u_b_arr = np.logspace(-2, 5, 1000)  # m/yr

# Three effective pressures
N_vals = [1.0e6, 0.3e6, 0.05e6]  # Pa
N_labels = ['$N = 1$ MPa', '$N = 0.3$ MPa', '$N = 0.05$ MPa']
N_colors = ['#1f77b4', '#2ca02c', '#d62728']

fig, ax = plt.subplots(figsize=(7, 5.5))

for N_val, lab, col in zip(N_vals, N_labels, N_colors):
    tau = tau_b_schoof(u_b_arr, N_val, C_s, n_sl, A_sl, lambda_max)
    ax.loglog(u_b_arr, tau / 1e3, '-', color=col, lw=2.5, label=lab)

    # Iken's bound
    iken = C_s * N_val
    ax.axhline(iken / 1e3, color=col, ls=':', lw=1, alpha=0.5)

# Weertman law for reference (calibrated to match N=1 MPa at low velocity)
C_w_ref = 1e-20  # tuned for visual match
tau_w = tau_b_weertman(u_b_arr, C_w_ref, n_sl)
ax.loglog(u_b_arr, tau_w / 1e3, 'k--', lw=1.5, alpha=0.4,
          label='Weertman (no cavity limit)')

# Annotations
ax.annotate('Weertman\nregime', xy=(0.3, 5), fontsize=9,
            ha='center', va='center', color='0.4', style='italic')
ax.annotate('Coulomb\nregime', xy=(3e3, 200), fontsize=9,
            ha='center', va='center', color='0.4', style='italic')
ax.annotate('$\\tau_b \\to C_s N$\n(Iken bound)', xy=(3e4, 15),
            fontsize=8, ha='center', color='#d62728', style='italic')

ax.set_xlabel('Slip velocity $u_b$ (m yr$^{-1}$)', fontsize=12)
ax.set_ylabel('Basal drag $\\tau_b$ (kPa)', fontsize=12)
ax.set_xlim(0.01, 1e5)
ax.set_ylim(0.5, 1000)
ax.legend(loc='upper left', fontsize=9, frameon=False)
ax.set_title('Regularized Coulomb sliding law', fontsize=12)

fig.tight_layout()
fig.savefig('figures/fig02_drag_vs_slip_velocity.pdf', bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

---
## Figure 3: Sliding Velocity vs Effective Pressure

For a fixed basal shear stress, show how the sliding velocity depends on effective pressure $N = P_i - P_w$ using the Budd-type law $u_b = k \tau_b^p / N^q$. This illustrates the extreme sensitivity of sliding to water pressure.

In [ ]:
# ---- Fig 3: sliding velocity vs effective pressure ----

H = 500.0          # m, ice thickness
P_i = rho_i * g * H  # Pa

# Effective pressure as fraction of overburden
Pw_frac = np.linspace(0.5, 0.995, 500)
N_arr = P_i * (1 - Pw_frac)  # Pa

tau_b_vals = [50e3, 100e3, 200e3]  # Pa
tau_labels = ['$\\tau_b = 50$ kPa', '$\\tau_b = 100$ kPa', '$\\tau_b = 200$ kPa']
tau_colors = ['#2ca02c', '#1f77b4', '#d62728']

# Budd-type law: u_b = k * tau_b^3 / N  (p=3, q=1)
k_budd = 3.5e-16  # m Pa^{-2} s^{-1} (representative)

fig, ax = plt.subplots(figsize=(7, 5.5))

for tau_val, lab, col in zip(tau_b_vals, tau_labels, tau_colors):
    u_b = k_budd * tau_val**3 / N_arr  # m/s
    u_b_yr = u_b * sec_per_yr  # m/yr
    ax.semilogy(Pw_frac * 100, u_b_yr, '-', color=col, lw=2.5, label=lab)

# Mark key thresholds
ax.axvline(93, color='gray', ls=':', lw=1, alpha=0.5)
ax.text(93.5, 0.5, '$P_s$ (cavity\nonset)', fontsize=8, color='gray',
        va='bottom', ha='left', style='italic')

ax.axvline(96, color='gray', ls='--', lw=1, alpha=0.5)
ax.text(96.5, 0.5, '$P_c$ (Iken\nbound)', fontsize=8, color='gray',
        va='bottom', ha='left', style='italic')

ax.set_xlabel('Water pressure $P_w / P_i$ (%)', fontsize=12)
ax.set_ylabel('Sliding velocity $u_b$ (m yr$^{-1}$)', fontsize=12)
ax.set_title(f'$H = {H:.0f}$ m, Budd law: $u_b = k\\,\\tau_b^3 / N$', fontsize=12)
ax.set_xlim(50, 100)
ax.set_ylim(0.1, 1e4)
ax.legend(loc='upper left', fontsize=10, frameon=False)

fig.tight_layout()
fig.savefig('figures/fig03_velocity_vs_effective_pressure.pdf', bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

---
## Figure 4: Mohr–Coulomb Yield Stress vs Effective Pressure

Show the linear relation $\tau_* = c_o + f N$ for tills with different friction angles, alongside measured values from Table 7.5 of Cuffey & Paterson.

In [ ]:
# ---- Fig 4: Mohr-Coulomb yield stress vs effective pressure ----

N_mc = np.linspace(0, 200, 500)  # kPa

# Friction angles and cohesion from Table 7.5 (Cuffey & Paterson)
tills = [
    {'name': 'Storglaciären', 'phi': 26, 'c_o': 5, 'color': '#1f77b4'},
    {'name': 'Black Rapids', 'phi': 40, 'c_o': 1.3, 'color': '#d62728'},
    {'name': 'Breidamerkur', 'phi': 32, 'c_o': 3.7, 'color': '#2ca02c'},
    {'name': 'Two Rivers', 'phi': 18, 'c_o': 14, 'color': '#ff7f0e'},
]

fig, ax = plt.subplots(figsize=(7, 5))

for till in tills:
    f_val = np.tan(np.radians(till['phi']))
    tau_star = till['c_o'] + f_val * N_mc
    ax.plot(N_mc, tau_star, '-', color=till['color'], lw=2,
            label=f"{till['name']} ($\\varphi = {till['phi']}°$)")
    # Mark the N=50 kPa reference point
    tau_50 = till['c_o'] + f_val * 50
    ax.plot(50, tau_50, 'o', color=till['color'], ms=8,
            markeredgecolor='k', markeredgewidth=0.6, zorder=5)

# Reference: typical glacier basal shear stress
ax.axhline(100, color='gray', ls=':', lw=1, alpha=0.5)
ax.text(5, 103, 'Typical $\\tau_b \\approx 100$ kPa', fontsize=9,
        color='gray', style='italic')

ax.set_xlabel('Effective normal stress $N$ (kPa)', fontsize=12)
ax.set_ylabel('Yield stress $\\tau_*$ (kPa)', fontsize=12)
ax.set_title('Mohr–Coulomb: $\\tau_* = c_o + N \\tan\\varphi$', fontsize=12)
ax.set_xlim(0, 200)
ax.set_ylim(0, 200)
ax.legend(loc='upper left', fontsize=9, frameon=False)

fig.tight_layout()
fig.savefig('figures/fig04_mohr_coulomb.pdf', bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

---
## Figure 5: Comparison of Sliding Laws

Show the three sliding laws side by side — Weertman, Budd (effective-pressure-dependent), and regularized Coulomb — plotted as $\tau_b$ vs $u_b$ for a fixed effective pressure. This illustrates the qualitative differences: Weertman grows without bound, Budd grows without bound (but shifted by $N$), and Coulomb saturates at $fN$.

In [ ]:
# ---- Fig 5: comparison of three sliding laws ----

u_b_comp = np.logspace(-2, 5, 1000)  # m/yr
N_comp = 0.2e6  # Pa (200 kPa, moderate effective pressure)

n_c = 3

# 1. Weertman: tau_b = C_w^{-1/m} * u_b^{1/m}
#    Calibrate C_w so that tau_b ~ 100 kPa at u_b ~ 100 m/yr
u_ref = 100 / sec_per_yr  # m/s
tau_ref = 100e3  # Pa
C_w = (u_ref / tau_ref**n_c)  # m/s Pa^{-3}
tau_weertman = (u_b_comp / sec_per_yr / C_w)**(1.0/n_c)

# 2. Budd: tau_b = (u_b * N^q / k)^{1/p}
#    u_b = k * tau_b^p / N^q, with p=3, q=1
#    Calibrate k so same reference point
k_b = u_ref * N_comp / tau_ref**3
tau_budd = (u_b_comp / sec_per_yr * N_comp / k_b)**(1.0/3)

# 3. Regularized Coulomb (Schoof 2005)
C_s_comp = 0.5
lambda_max_comp = 0.5
tau_coulomb = tau_b_schoof(u_b_comp, N_comp, C_s_comp, n_c, A_sl, lambda_max_comp)

fig, ax = plt.subplots(figsize=(7, 5.5))

ax.loglog(u_b_comp, tau_weertman / 1e3, '--', color='#1f77b4', lw=2,
          label='Weertman: $\\tau_b = C^{-1/m}\\,u_b^{1/m}$')
ax.loglog(u_b_comp, tau_budd / 1e3, '-.', color='#2ca02c', lw=2,
          label='Budd: $\\tau_b \\propto (u_b\\,N)^{1/p}$')
ax.loglog(u_b_comp, tau_coulomb / 1e3, '-', color='#d62728', lw=2.5,
          label='Regularized Coulomb')

# Iken/Coulomb bound
iken_bound = C_s_comp * N_comp
ax.axhline(iken_bound / 1e3, color='#d62728', ls=':', lw=1, alpha=0.5)
ax.text(2e4, iken_bound / 1e3 * 1.15, f'$f N = {iken_bound/1e3:.0f}$ kPa (Iken bound)',
        fontsize=9, color='#d62728', ha='right', va='bottom', style='italic')

ax.set_xlabel('Slip velocity $u_b$ (m yr$^{-1}$)', fontsize=12)
ax.set_ylabel('Basal drag $\\tau_b$ (kPa)', fontsize=12)
ax.set_title(f'Comparison of sliding laws ($N = {N_comp/1e3:.0f}$ kPa)', fontsize=12)
ax.set_xlim(0.01, 1e5)
ax.set_ylim(1, 1000)
ax.legend(loc='lower right', fontsize=9, frameon=False)

fig.tight_layout()
fig.savefig('figures/fig05_sliding_law_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

---
## Figure 6: Regelation and Enhanced Creep Schematic

A physical diagram showing how ice moves past a bedrock bump by two mechanisms: regelation (melting on the upstream side, refreezing on the downstream side, with heat conducted through the bump) and enhanced creep (viscous deformation of ice around the bump). This replaces Chalkboard Drawing 1 in the tex.

In [ ]:
# ---- Fig 6: regelation and enhanced creep schematic ----
from matplotlib.patches import FancyArrowPatch, Arc

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax_idx, (ax, title) in enumerate(zip(axes, ['(a) Regelation', '(b) Enhanced creep'])):
    ax.set_xlim(-0.5, 5.5)
    ax.set_ylim(-1.2, 3.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)

    # Draw bedrock bump (Gaussian shape)
    x_bed = np.linspace(-0.5, 5.5, 500)
    bump = 1.2 * np.exp(-((x_bed - 2.5) / 0.8)**2)
    ax.fill_between(x_bed, bump, -1.2, color='#8B7355', alpha=0.5)
    ax.plot(x_bed, bump, 'k-', lw=2)
    ax.text(2.5, -0.7, 'bedrock', fontsize=10, ha='center', color='#5C4033',
            fontweight='bold')

    # Ice region
    ax.fill_between(x_bed, bump, 3.5, color='#b3d9ff', alpha=0.25)
    ax.text(4.5, 2.8, 'ice', fontsize=11, ha='center', color='#2060a0',
            fontweight='bold')

    # Ice flow arrow
    ax.annotate('', xy=(4.8, 2.2), xytext=(0.5, 2.2),
                arrowprops=dict(arrowstyle='->', lw=2.5, color='#2060a0'))
    ax.text(2.65, 2.5, 'ice flow', fontsize=10, ha='center', color='#2060a0')

    if ax_idx == 0:  # Regelation
        # High pressure upstream
        ax.annotate('high $P$\n(melting)', xy=(1.5, 0.55), fontsize=9,
                    ha='center', color='#d62728', fontweight='bold')
        # Low pressure downstream
        ax.annotate('low $P$\n(refreezing)', xy=(3.5, 0.55), fontsize=9,
                    ha='center', color='#1f77b4', fontweight='bold')
        # Meltwater flow arrow around bump
        ax.annotate('', xy=(3.3, -0.15), xytext=(1.7, -0.15),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='#4a90d9',
                                    connectionstyle='arc3,rad=-0.3'))
        ax.text(2.5, -0.45, 'meltwater', fontsize=8, ha='center',
                color='#4a90d9', style='italic')
        # Heat conduction arrow through bump
        ax.annotate('', xy=(1.6, 0.2), xytext=(3.4, 0.2),
                    arrowprops=dict(arrowstyle='->', lw=1.5, color='#ff7f0e',
                                    connectionstyle='arc3,rad=0.2'))
        ax.text(2.5, 0.55, 'heat', fontsize=8, ha='center',
                color='#ff7f0e', style='italic', rotation=0)
        ax.text(2.5, -1.05, 'Effective for small bumps ($u_1 \\propto 1/a$)',
                fontsize=9, ha='center', color='0.3', style='italic')
    else:  # Enhanced creep
        # Flowlines bending around bump
        for y_off in [0.3, 0.7, 1.1, 1.5]:
            x_flow = np.linspace(0, 5, 200)
            bump_flow = 1.2 * np.exp(-((x_flow - 2.5) / 0.8)**2)
            deflection = 0.5 * y_off * np.exp(-((x_flow - 2.5) / 1.0)**2)
            y_flow = bump_flow + y_off + deflection
            ax.plot(x_flow, y_flow, '-', color='#2060a0', lw=0.8, alpha=0.5)
            # Small arrow in the middle
            mid = len(x_flow) // 2 + 20
            ax.annotate('', xy=(x_flow[mid], y_flow[mid]),
                        xytext=(x_flow[mid-8], y_flow[mid-8]),
                        arrowprops=dict(arrowstyle='->', color='#2060a0',
                                        lw=0.8, alpha=0.5))
        # Stress concentration annotation
        ax.annotate('stress $\\sim \\tau_b / R^2$', xy=(1.3, 1.3), fontsize=9,
                    ha='center', color='#d62728', fontweight='bold')
        ax.annotate('', xy=(1.7, 0.9), xytext=(1.3, 1.2),
                    arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.2))
        ax.text(2.5, -1.05, 'Effective for large bumps ($u_2 \\propto a$)',
                fontsize=9, ha='center', color='0.3', style='italic')

fig.tight_layout()
fig.savefig('figures/fig06_regelation_creep_schematic.pdf', bbox_inches='tight')
plt.show()
print('Figure 6 saved.')

---
## Figure 7: Cavity Formation on a Sinusoidal Bed

Show three stages of cavity growth as water pressure increases: (a) full contact, (b) small cavities in the lee of bumps, (c) extensive cavitation with ice contact only on upstream faces. This makes the abstract concept of cavitation concrete.

In [ ]:
# ---- Fig 7: cavity formation on a sinusoidal bed ----

x = np.linspace(0, 3, 1000)  # three wavelengths
bed = 0.3 * np.sin(2 * np.pi * x)

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)

titles = [
    '(a) Full contact: $P_w < P_s$',
    '(b) Small cavities: $P_w \\gtrsim P_s$',
    '(c) Extensive cavitation: $P_w \\to P_c$',
]
cavity_fracs = [0.0, 0.3, 0.7]  # fraction of lee side that is cavitated

for ax, title, cav_frac in zip(axes, titles, cavity_fracs):
    ax.set_ylim(-0.8, 1.5)
    ax.set_xlim(0, 3)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=11, loc='left', fontweight='bold')

    # Draw bedrock
    ax.fill_between(x, bed, -0.8, color='#8B7355', alpha=0.4)
    ax.plot(x, bed, 'k-', lw=1.5)

    if cav_frac == 0:
        # Full ice-bed contact
        ice_base = bed.copy()
    else:
        # Create cavities on downstream (lee) side of each bump
        ice_base = bed.copy()
        for peak_x in [0.25, 1.25, 2.25]:
            # The peak of each sine is at x = 0.25 + n
            # Lee side extends from peak to trough (peak_x to peak_x + 0.5)
            cav_start = peak_x
            cav_end = peak_x + 0.5 * cav_frac + 0.15
            mask = (x >= cav_start + 0.05) & (x <= cav_end)
            # Cavity roof: straight line from detachment to reattachment
            x_det = cav_start + 0.05
            x_reatt = cav_end
            bed_det = 0.3 * np.sin(2 * np.pi * x_det)
            bed_reatt = 0.3 * np.sin(2 * np.pi * x_reatt)
            roof_line = bed_det + (bed_reatt - bed_det) * (x[mask] - x_det) / (x_reatt - x_det)
            # Only replace where roof is above bed (actual cavity)
            cavity_exists = roof_line > bed[mask]
            ice_base_section = ice_base[mask].copy()
            ice_base_section[cavity_exists] = roof_line[cavity_exists]
            ice_base[mask] = ice_base_section

    # Fill ice
    ax.fill_between(x, ice_base, 1.5, color='#b3d9ff', alpha=0.3)

    # Fill cavities (water) where ice_base > bed
    cavity_mask = ice_base > bed + 0.005
    if np.any(cavity_mask):
        ax.fill_between(x, bed, ice_base, where=cavity_mask,
                         color='#4a90d9', alpha=0.3)
        # Label one cavity
        first_cav = np.where(cavity_mask)[0]
        if len(first_cav) > 10:
            mid_cav = first_cav[len(first_cav) // 2]
            cav_y = 0.5 * (bed[mid_cav] + ice_base[mid_cav])
            ax.text(x[mid_cav], cav_y, 'cavity', fontsize=8,
                    ha='center', va='center', color='#2060a0', style='italic')

    # Ice flow arrow
    ax.annotate('', xy=(2.8, 1.1), xytext=(0.2, 1.1),
                arrowprops=dict(arrowstyle='->', lw=2, color='#2060a0'))
    ax.text(1.5, 1.25, 'ice flow', fontsize=9, ha='center', color='#2060a0')

    # Draw ice base line
    ax.plot(x, ice_base, '-', color='#2060a0', lw=1, alpha=0.7)

    # Normal stress arrows on contact areas (upstream sides)
    if cav_frac > 0:
        for peak_x in [0.25, 1.25, 2.25]:
            # Upstream contact: arrows pointing into the bed
            contact_x = peak_x - 0.15
            contact_y = 0.3 * np.sin(2 * np.pi * contact_x)
            ax.annotate('', xy=(contact_x - 0.05, contact_y + 0.05),
                        xytext=(contact_x - 0.15, contact_y + 0.25),
                        arrowprops=dict(arrowstyle='->', color='#d62728',
                                        lw=1.2))

axes[-1].text(1.5, -0.65, 'bedrock', fontsize=10, ha='center',
              color='#5C4033', fontweight='bold')

fig.tight_layout()
fig.savefig('figures/fig07_cavity_formation.pdf', bbox_inches='tight')
plt.show()
print('Figure 7 saved.')

---
## Figure 8: Form Drag vs Skin Friction — Annotated Sliding Law

Show the regularized Coulomb sliding law with clear annotations identifying the form-drag regime (low velocity, rate-strengthening) and skin-friction regime (high velocity, rate-independent), following Minchew & Joughin (2020) and Zoet & Iverson (2020). This is a pedagogical companion to Figure 2.

In [ ]:
# ---- Fig 8: form drag vs skin friction annotated sliding law ----

N_fig8 = 0.3e6  # Pa
u_b_fig8 = np.logspace(-1, 4.5, 1000)  # m/yr
C_s_fig8 = 0.5
n_fig8 = 3

tau_fig8 = tau_b_schoof(u_b_fig8, N_fig8, C_s_fig8, n_fig8, A_sl, 0.5)

# Find transition velocity (where tau reaches ~90% of Coulomb limit)
coulomb_limit = C_s_fig8 * N_fig8
idx_trans = np.argmin(np.abs(tau_fig8 - 0.85 * coulomb_limit))
u_trans = u_b_fig8[idx_trans]

fig, ax = plt.subplots(figsize=(8, 5))

ax.semilogx(u_b_fig8, tau_fig8 / 1e3, 'k-', lw=3)

# Coulomb limit
ax.axhline(coulomb_limit / 1e3, color='#d62728', ls='--', lw=1.5, alpha=0.7)
ax.text(2e4, coulomb_limit / 1e3 + 3, f'Coulomb limit: $fN = {coulomb_limit/1e3:.0f}$ kPa',
        fontsize=10, ha='right', color='#d62728')

# Transition velocity
ax.axvline(u_trans, color='gray', ls=':', lw=1, alpha=0.5)
ax.text(u_trans * 0.7, 20, f'$u_t \\approx {u_trans:.0f}$ m/yr', fontsize=9,
        ha='right', color='0.4', rotation=90, va='bottom')

# Shade form-drag regime
ax.axvspan(0.1, u_trans, alpha=0.06, color='#1f77b4')
ax.text(3, 50, 'Form-drag regime\n(rate-strengthening)',
        fontsize=11, ha='center', va='center', color='#1f77b4',
        fontweight='bold', style='italic')
ax.text(3, 35, '$\\tau_b \\propto u_b^{1/n}$\nDrag from viscous flow\naround bed obstacles',
        fontsize=9, ha='center', va='center', color='#1f77b4')

# Shade skin-friction regime
ax.axvspan(u_trans, 3e4, alpha=0.06, color='#d62728')
ax.text(5000, 50, 'Skin-friction regime\n(rate-independent)',
        fontsize=11, ha='center', va='center', color='#d62728',
        fontweight='bold', style='italic')
ax.text(5000, 35, '$\\tau_b \\to fN$\nDrag from Coulomb friction\nat grain contacts',
        fontsize=9, ha='center', va='center', color='#d62728')

# Arrow showing the transition
ax.annotate('transition', xy=(u_trans, 120), xytext=(u_trans * 5, 110),
            fontsize=9, ha='left', color='0.3',
            arrowprops=dict(arrowstyle='->', color='0.3', lw=1.2))

ax.set_xlabel('Slip velocity $u_b$ (m yr$^{-1}$)', fontsize=12)
ax.set_ylabel('Basal drag $\\tau_b$ (kPa)', fontsize=12)
ax.set_xlim(0.1, 3e4)
ax.set_ylim(0, 170)
ax.set_title(f'Two regimes of basal drag ($N = {N_fig8/1e3:.0f}$ kPa)', fontsize=12)

fig.tight_layout()
fig.savefig('figures/fig08_form_drag_skin_friction.pdf', bbox_inches='tight')
plt.show()
print('Figure 8 saved.')

---
## Figure 9: UPB Model — Melt Rate and Ice Stream Velocity vs Bed Strength

Two-panel figure from the Tulaczyk et al. (2000b) undrained plastic bed model. (a) Basal melt rate as a function of bed strength $\tau_b/\tau_d$, showing the maximum at $\tau_b = 0.25\,\tau_d$ and the two equilibria. (b) Ice stream velocity vs bed strength, with stable and unstable equilibria marked and arrows showing the direction of evolution.

In [ ]:
# ---- Fig 9: UPB stability diagram (Tulaczyk et al. 2000b) ----

# Parameters for Ice Stream B (UpB camp)
tau_d = 13.0    # kPa, driving stress
n_upb = 3       # Glen's flow law exponent
W = 17.0        # ice stream half-width in units of ice thickness
G = 0.06        # W m^{-2}, geothermal flux
k_i = 2.1       # W m^{-1} K^{-1}, thermal conductivity of ice
theta_b = 0.04  # K m^{-1}, basal temperature gradient
L_i = 333.5e3   # J kg^{-1}, latent heat
rho_ice = 900.0 # kg m^{-3}
B_n = 1.45e-16  # Pa^{-3} s^{-1} (rate factor for T = -15 C, from Paterson)
H_upb = 1000.0  # m, ice thickness

# Compute U_d (deformation velocity)
U_d = 2**(1-n_upb) * (tau_d * 1e3)**n_upb * H_upb / ((n_upb + 1) * (1 / B_n))
U_d_yr = U_d * sec_per_yr  # very small for ISB

# Ice stream velocity as function of bed strength (equation 5 in Tulaczyk 2000b)
tau_b_range = np.linspace(0.001, tau_d * 0.999, 500)  # kPa
U_b = ((1 - tau_b_range / tau_d)**n_upb) * W**(n_upb + 1) * U_d_yr  # m/yr

# Shear heating: tau_b * U_b (kPa * m/yr -> W/m^2)
shear_heating = tau_b_range * 1e3 * U_b / sec_per_yr  # Pa * m/s = W/m^2

# Melt rate (equation 1 / 6 in Tulaczyk 2000b)
conductive_loss = k_i * theta_b  # W/m^2
m_r = (shear_heating + G - conductive_loss) / (L_i * rho_ice)  # m/s
m_r_yr = m_r * sec_per_yr * 1e3  # mm/yr

# Find equilibria (where m_r = 0)
sign_changes = np.where(np.diff(np.sign(m_r_yr)))[0]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8), sharex=True)

# ---- Panel A: Melt rate vs bed strength ----
ax1.plot(tau_b_range / tau_d, m_r_yr, 'k-', lw=2.5)
ax1.axhline(0, color='gray', ls='-', lw=0.5)
ax1.axvline(0.25, color='gray', ls=':', lw=1, alpha=0.5)
ax1.text(0.26, max(m_r_yr) * 0.85,
         '$\\tau_b = \\tau_d / (n+1)$\n(max shear heating)',
         fontsize=9, color='0.4', va='top')

# Mark equilibria
for idx in sign_changes:
    tau_eq = tau_b_range[idx] / tau_d
    mr_eq = 0
    if tau_eq < 0.25:
        ax1.plot(tau_eq, mr_eq, 'o', color='k', ms=12, zorder=5)
        ax1.text(tau_eq + 0.03, 0.3, 'stable\n(ice stream)', fontsize=9,
                 color='#2ca02c', fontweight='bold', va='bottom')
    else:
        ax1.plot(tau_eq, mr_eq, 'o', color='white', ms=12, zorder=5,
                 markeredgecolor='k', markeredgewidth=2)
        ax1.text(tau_eq + 0.03, 0.3, 'unstable', fontsize=9,
                 color='#d62728', fontweight='bold', va='bottom')

# Shade melting/freezing regions
ax1.fill_between(tau_b_range / tau_d, m_r_yr, 0,
                  where=m_r_yr > 0, alpha=0.08, color='#d62728')
ax1.fill_between(tau_b_range / tau_d, m_r_yr, 0,
                  where=m_r_yr < 0, alpha=0.08, color='#1f77b4')
ax1.text(0.05, max(m_r_yr) * 0.3, 'melting', fontsize=10, color='#d62728',
         style='italic')
if min(m_r_yr) < -0.1:
    ax1.text(0.8, min(m_r_yr) * 0.5, 'freezing', fontsize=10, color='#1f77b4',
             style='italic')

# Arrows showing direction of evolution
for x_arr, dir_sgn in [(0.07, -1), (0.5, 1), (0.85, -1)]:
    idx_arr = np.argmin(np.abs(tau_b_range / tau_d - x_arr))
    if m_r_yr[idx_arr] > 0:
        # Melting -> weakening -> tau_b decreases
        ax1.annotate('', xy=(x_arr - 0.04, m_r_yr[idx_arr] * 0.5),
                      xytext=(x_arr + 0.04, m_r_yr[idx_arr] * 0.5),
                      arrowprops=dict(arrowstyle='->', color='0.5', lw=1.5))
    elif m_r_yr[idx_arr] < 0:
        # Freezing -> strengthening -> tau_b increases
        ax1.annotate('', xy=(x_arr + 0.04, m_r_yr[idx_arr] * 0.5),
                      xytext=(x_arr - 0.04, m_r_yr[idx_arr] * 0.5),
                      arrowprops=dict(arrowstyle='->', color='0.5', lw=1.5))

ax1.set_ylabel('Basal melt rate $m_r$ (mm yr$^{-1}$)', fontsize=12)
ax1.set_title('(a) Basal energy balance', fontsize=12, loc='left', fontweight='bold')

# ---- Panel B: Velocity vs bed strength ----
ax2.plot(tau_b_range / tau_d, U_b, 'k-', lw=2.5)

# Mark same equilibria
for idx in sign_changes:
    tau_eq = tau_b_range[idx] / tau_d
    U_eq_idx = np.argmin(np.abs(tau_b_range / tau_d - tau_eq))
    U_eq = U_b[U_eq_idx]
    if tau_eq < 0.25:
        ax2.plot(tau_eq, U_eq, 'o', color='k', ms=12, zorder=5)
    else:
        ax2.plot(tau_eq, U_eq, 'o', color='white', ms=12, zorder=5,
                 markeredgecolor='k', markeredgewidth=2)

# Shade ice stream vs ice sheet modes
ax2.axvspan(0, 0.25, alpha=0.06, color='#2ca02c')
ax2.text(0.12, max(U_b) * 0.6, 'ice stream\nmode', fontsize=11,
         ha='center', color='#2ca02c', fontweight='bold', style='italic')
ax2.axvspan(0.25, 1.0, alpha=0.06, color='#9467bd')
ax2.text(0.65, max(U_b) * 0.6, 'ice sheet\nmode', fontsize=11,
         ha='center', color='#9467bd', fontweight='bold', style='italic')

ax2.set_xlabel('Bed strength $\\tau_b / \\tau_d$', fontsize=12)
ax2.set_ylabel('Ice stream velocity $U_b$ (m yr$^{-1}$)', fontsize=12)
ax2.set_title('(b) Ice stream velocity', fontsize=12, loc='left', fontweight='bold')
ax2.set_xlim(0, 1)

fig.tight_layout()
fig.savefig('figures/fig09_upb_stability.pdf', bbox_inches='tight')
plt.show()
print('Figure 9 saved.')

---
## Figure 10: Till Strength vs Porosity

Show the relationship $\tau_f = a \exp(-b\,n_p/(1-n_p))$ from Tulaczyk et al. (2000a), illustrating how extremely sensitive till strength is to small changes in porosity. This is the key relationship that makes the UPB feedback loop work.

In [ ]:
# ---- Fig 10: till strength vs porosity ----

# Parameters from Tulaczyk et al. (2000a), equation (3d)
# Original: tau_f = a * exp(-b * e), with e = phi_p / (1 - phi_p)
a_till = 944000  # kPa
b_till = 21.7

phi_p_range = np.linspace(0.20, 0.44, 500)
e_from_phi = phi_p_range / (1 - phi_p_range)  # void ratio
tau_f = a_till * np.exp(-b_till * e_from_phi)  # kPa

fig, ax1 = plt.subplots(figsize=(7, 5.5))

ax1.semilogy(phi_p_range * 100, tau_f, 'k-', lw=2.5)

# Mark observed UpB conditions: e ~ 0.6 -> phi_p = 0.6/1.6 = 0.375
phi_p_upb = 0.375
e_upb = phi_p_upb / (1 - phi_p_upb)
tau_upb = a_till * np.exp(-b_till * e_upb)
ax1.plot(phi_p_upb * 100, tau_upb, 'o', color='#d62728', ms=12, zorder=5,
         markeredgecolor='k', markeredgewidth=1)
ax1.annotate(f'UpB observed\n$\\phi_p \\approx {phi_p_upb*100:.0f}$%, '
             f'$\\tau_f \\approx {tau_upb:.1f}$ kPa',
             xy=(phi_p_upb * 100, tau_upb),
             xytext=(phi_p_upb * 100 - 8, tau_upb * 5),
             fontsize=10, ha='center', color='#d62728', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#d62728', lw=1.5))

# Show sensitivity: annotate a ~4 percentage point change in porosity
phi_p_low = 0.355   # e = 0.55
phi_p_high = 0.394   # e = 0.65
tau_low = a_till * np.exp(-b_till * phi_p_low / (1 - phi_p_low))
tau_high = a_till * np.exp(-b_till * phi_p_high / (1 - phi_p_high))
ax1.annotate('', xy=(phi_p_low * 100, tau_low),
             xytext=(phi_p_high * 100, tau_high),
             arrowprops=dict(arrowstyle='<->', color='#1f77b4', lw=2))
ax1.text(0.5 * (phi_p_low + phi_p_high) * 100 - 5,
         np.sqrt(tau_low * tau_high),
         f'$\\Delta \\phi_p = {(phi_p_high - phi_p_low)*100:.0f}$%\n'
         f'$\\tau_f$ changes\nby $\\times${tau_low/tau_high:.0f}',
         fontsize=9, ha='center', va='center', color='#1f77b4',
         fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                   edgecolor='#1f77b4', alpha=0.9))

# Shade ice-stream-relevant range
ax1.axhspan(0.5, 10, alpha=0.06, color='#2ca02c')
ax1.text(22, 3, 'ice stream range\n($\\tau_f < 10$ kPa)', fontsize=9,
         color='#2ca02c', style='italic')

# Reference lines
ax1.axhline(13, color='gray', ls=':', lw=1, alpha=0.5)
ax1.text(21, 14, '$\\tau_d \\approx 13$ kPa (ISB driving stress)',
         fontsize=8, color='0.5', style='italic')

ax1.set_xlabel('Porosity $\\phi_p$ (%)', fontsize=12)
ax1.set_ylabel('Till failure strength $\\tau_f$ (kPa)', fontsize=12)
ax1.set_xlim(20, 44)
ax1.set_ylim(0.5, 500)
ax1.set_title('$\\tau_f = a\\,\\exp\\!\\left(-b\\,\\phi_p / (1 - \\phi_p)\\right)$',
              fontsize=12)

fig.tight_layout()
fig.savefig('figures/fig10_till_strength_porosity.pdf', bbox_inches='tight')
plt.show()
print('Figure 10 saved.')